# 02 - Metadata Anatomy

## What you'll learn
- How `metadata.json` points to snapshots and manifest lists.
- How manifest lists point to manifests.
- How manifests point to physical data files and column stats.

## Interview questions this answers
- Walk me through what happens when you query an Iceberg table.
- How does Iceberg prune files at query time?
- What's the difference between a manifest list and a manifest?

In [ ]:
import json
from pathlib import Path

from fastavro import reader

from src.catalog_helper import current_metadata_location, download_s3_uri, get_catalog, read_json_s3

catalog = get_catalog()
table = catalog.load_table("lab.events")

Iceberg queries start at the current table metadata file. Do not hard-code `v1.metadata.json`: after a write, the useful snapshot data is usually in a later metadata version.

In [ ]:
metadata_uri = current_metadata_location(table)
metadata = read_json_s3(metadata_uri)
print(metadata_uri)
print(json.dumps({key: metadata.get(key) for key in [
    "format-version", "table-uuid", "location", "last-sequence-number",
    "current-schema-id", "current-snapshot-id"
]}, indent=2))

Top-level keys are the table's control plane. The data plane is reached through snapshots and manifests.

In [ ]:
for key in metadata.keys():
    value = metadata[key]
    summary = f"list[{len(value)}]" if isinstance(value, list) else f"dict[{len(value)}]" if isinstance(value, dict) else value
    print(f"{key}: {summary}")

Follow the current snapshot to its manifest list Avro file.

In [ ]:
current_snapshot_id = metadata["current-snapshot-id"]
snapshot = next(item for item in metadata["snapshots"] if item["snapshot-id"] == current_snapshot_id)
manifest_list_uri = snapshot["manifest-list"]
manifest_list_path = download_s3_uri(manifest_list_uri, Path("/tmp/iceberg-lab/manifest-list.avro"))

with manifest_list_path.open("rb") as fh:
    manifest_list_rows = list(reader(fh))

print("Manifest list:", manifest_list_uri)
print("Rows:", len(manifest_list_rows))
manifest_list_rows

Now follow one manifest entry. This is where file-level stats appear.

In [ ]:
manifest_uri = manifest_list_rows[0]["manifest_path"]
manifest_path = download_s3_uri(manifest_uri, Path("/tmp/iceberg-lab/manifest.avro"))

with manifest_path.open("rb") as fh:
    manifest_rows = list(reader(fh))

first = manifest_rows[0]
data_file = first["data_file"]
print("Manifest:", manifest_uri)
print("Status:", first.get("status"))
print("Data file:", data_file.get("file_path"))
print("Record count:", data_file.get("record_count"))
print("Column lower bounds keys:", list((data_file.get("lower_bounds") or {}).keys()))
print("Column upper bounds keys:", list((data_file.get("upper_bounds") or {}).keys()))

Pointer chain:

```text
metadata.json -> manifest list -> manifest -> data files
      |                |              |
      |                |              +-- file path, row count, bounds, null counts
      |                +-- which manifests changed in a snapshot
      +-- schemas, partition specs, snapshots, refs, table properties
```

## Try yourself
- Append another small batch in notebook 01 and rerun this notebook. Which files are new?
- Pick a column bound in the manifest and explain how it could help skip a file.